# 🔍 Day 4: RAG Systems - Chat with Your Documents

## ITI - AEC Informatics Track | Gen AI Course

---

### 📋 المحتوى:

| # | الموضوع | الوقت |
|---|---------|-------|
| 0 | التجهيزات والمكتبات | 5 min |
| 1 | مفهوم RAG | 10 min |
| 2 | Text Embeddings | 15 min |
| 3 | Vector Database | 15 min |
| 4 | Document Chunking | 10 min |
| 5 | بناء RAG كامل | 20 min |
| 6 | Streamlit App | 15 min |
| 7 | 🔬 Lab Project | 60 min |

---

# 🔧 Section 0: التجهيزات

## ⚠️ مهم جداً: شغّل الـ cells بالترتيب!

In [1]:
!pip install tf-keras


In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  🔴 الخطوة 1: تثبيت كل المكتبات (شغّل مرة واحدة)             ║
# ║                                                              ║
# ║  بعد التثبيت: Restart Kernel ثم شغّل من الخطوة 2             ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("📦 Installing packages...")
print("   (ممكن تاخد دقيقة أو اتنين)")
print()

# NumPy fix (مهم!)
install("numpy<2")
print("✅ 1/6 NumPy")

# LangChain packages
install("langchain-google-genai")
print("✅ 2/6 langchain-google-genai")

install("langchain-community")
print("✅ 3/6 langchain-community")

install("langchain-text-splitters")
print("✅ 4/6 langchain-text-splitters")

# Vector DB & PDF
install("chromadb")
print("✅ 5/6 chromadb")

install("pypdf")
print("✅ 6/6 pypdf")

print()
print("="*50)
print("✅ كل المكتبات اتثبتت!")
print()
print("⚠️  دلوقتي اعمل: Kernel → Restart Kernel")
print("    وبعدين شغّل من الخطوة 2")
print("="*50)

📦 Installing packages...
   (ممكن تاخد دقيقة أو اتنين)

✅ 1/6 NumPy
✅ 2/6 langchain-google-genai
✅ 3/6 langchain-community
✅ 4/6 langchain-text-splitters
✅ 5/6 chromadb
✅ 6/6 pypdf

✅ كل المكتبات اتثبتت!

⚠️  دلوقتي اعمل: Kernel → Restart Kernel
    وبعدين شغّل من الخطوة 2


In [3]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  🟢 الخطوة 2: التحقق من المكتبات                             ║
# ╚══════════════════════════════════════════════════════════════╝

import numpy as np

print("🔍 Checking packages...")
print()

# Check NumPy
print(f"NumPy: {np.__version__}", end=" ")
if int(np.__version__.split('.')[0]) >= 2:
    print("⚠️ Warning: NumPy 2.x")
else:
    print("✅")

# Check LangChain packages (just import, no version)
try:
    import langchain_google_genai
    print("langchain-google-genai: ✅")
except ImportError:
    print("langchain-google-genai: ❌")

try:
    import langchain_community
    print("langchain-community: ✅")
except ImportError:
    print("langchain-community: ❌")

try:
    import langchain_text_splitters
    print("langchain-text-splitters: ✅")
except ImportError:
    print("langchain-text-splitters: ❌")

try:
    import chromadb
    print("chromadb: ✅")
except ImportError:
    print("chromadb: ❌")

try:
    import pypdf
    print("pypdf: ✅")
except ImportError:
    print("pypdf: ❌")

print()
print("=" * 50)
print("✅ جاهز! كمّل للخطوة التالية.")

🔍 Checking packages...

NumPy: 1.26.4 ✅


c:\Users\ibrah\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


langchain-google-genai: ✅
langchain-community: ✅

langchain-text-splitters: ✅
chromadb: ✅
pypdf: ✅

✅ جاهز! كمّل للخطوة التالية.


In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  🟢 الخطوة 3: API Key Helper                                 ║
# ╚══════════════════════════════════════════════════════════════╝

import os
from getpass import getpass

_API_KEYS = {}

def get_api_key(key_name: str, display_name: str = None) -> str:
    """
    بيجيب الـ API Key من:
    1. Colab Secrets (لو في Colab)
    2. Environment Variables
    3. .env file
    4. Manual input
    """
    if display_name is None:
        display_name = key_name
    
    # لو موجود في الـ cache
    if key_name in _API_KEYS and _API_KEYS[key_name]:
        return _API_KEYS[key_name]
    
    api_key = None
    source = None
    
    # Try Colab Secrets
    try:
        from google.colab import userdata
        api_key = userdata.get(key_name)
        if api_key:
            source = "Colab Secrets"
    except:
        pass
    
    # Try Environment Variable
    if not api_key:
        api_key = os.environ.get(key_name)
        if api_key:
            source = "Environment Variable"
    
    # Try .env file
    if not api_key:
        try:
            from dotenv import load_dotenv
            load_dotenv()
            api_key = os.environ.get(key_name)
            if api_key:
                source = ".env file"
        except ImportError:
            pass
    
    # Ask user
    if not api_key:
        print(f"🔑 {display_name} API Key مش موجود.")
        print(f"   احصل عليه من: https://aistudio.google.com/apikey")
        api_key = getpass(f"   ادخل {display_name} API Key: ")
        source = "Manual Input"
    
    _API_KEYS[key_name] = api_key
    print(f"✅ {display_name} API Key loaded from {source}")
    return api_key

print("✅ API Key Helper ready!")

✅ API Key Helper ready!


---

# 📚 Section 1: مفهوم RAG

## 🤔 إيه المشكلة؟

الـ LLM (زي ChatGPT أو Gemini) عندهم مشاكل:

| المشكلة | التفسير |
|---------|--------|
| **Knowledge Cutoff** | متدربين على data قديمة |
| **No Private Data** | مش عارفين documents الشركة |
| **Hallucinations** | بيألفوا لما مش عارفين |

---

## 💡 الحل: RAG!

**RAG = Retrieval-Augmented Generation**

```
┌─────────────────────────────────────────────────────────┐
│                                                         │
│   📄 PDF ──→ 🔪 Chunk ──→ 🔢 Embed ──→ 💾 Store        │
│                                         (Vector DB)     │
│                                              │          │
│   ❓ Question ──→ 🔢 Embed ──→ 🔍 Search ────┘          │
│                                    │                    │
│                                    ▼                    │
│                         📝 Relevant Chunks              │
│                                    │                    │
│                                    ▼                    │
│         Question + Chunks ──→ 🤖 LLM ──→ 💬 Answer     │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

---

# 📚 Section 2: Text Embeddings

## 🔢 إيه هو الـ Embedding?

**تحويل النص لـ Vector (قائمة أرقام)**

```
"BIM is Building Information Modeling"
                ↓
    [0.23, -0.45, 0.12, 0.89, ...]
            (768 رقم)
```

## 🎯 الفكرة العبقرية:

**النصوص المتشابهة في المعنى = Vectors قريبة من بعض!**

In [5]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  إعداد الـ Embedding Model                                   ║
# ║                                                              ║
# ║  ⚠️ مهم: الـ model name هو gemini-embedding-001              ║
# ║     (من غير models/ prefix)                                  ║
# ╚══════════════════════════════════════════════════════════════╝

from langchain_google_genai import GoogleGenerativeAIEmbeddings

# احصل على الـ API Key
GOOGLE_API_KEY = get_api_key('GOOGLE_API_KEY', 'Gemini')

# إنشاء الـ Embedding Model
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",   # ✅ الاسم الصحيح!
    google_api_key=GOOGLE_API_KEY
)

print("✅ Embedding Model ready!")
print("   Model: gemini-embedding-001")

🔑 Gemini API Key مش موجود.
   احصل عليه من: https://aistudio.google.com/apikey
✅ Gemini API Key loaded from Manual Input
✅ Embedding Model ready!
   Model: gemini-embedding-001


In [6]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  تجربة: تحويل نص لـ Vector                                   ║
# ╚══════════════════════════════════════════════════════════════╝

text = "BIM is Building Information Modeling"

print(f"📝 النص: {text}")
print("⏳ Converting to vector...")

vector = embeddings.embed_query(text)

print(f"\n✅ تم التحويل!")
print(f"📏 طول الـ Vector: {len(vector)} رقم")
print(f"🔢 أول 5 أرقام: {[round(x, 4) for x in vector[:5]]}")
print(f"🔢 آخر 5 أرقام: {[round(x, 4) for x in vector[-5:]]}")

📝 النص: BIM is Building Information Modeling
⏳ Converting to vector...

✅ تم التحويل!
📏 طول الـ Vector: 3072 رقم
🔢 أول 5 أرقام: [-0.0354, 0.0066, 0.0202, -0.0633, -0.0024]
🔢 آخر 5 أرقام: [-0.0091, 0.0131, -0.003, 0.0023, -0.0088]


In [7]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  تجربة: مقارنة التشابه (Cosine Similarity)                   ║
# ║                                                              ║
# ║  المعادلة:                                                   ║
# ║                      A · B                                   ║
# ║  Similarity = ─────────────────                              ║
# ║                ||A|| × ||B||                                 ║
# ║                                                              ║
# ║  A · B = Dot Product (حاصل الضرب النقطي)                     ║
# ║  ||A|| = طول الـ vector                                      ║
# ╚══════════════════════════════════════════════════════════════╝

import numpy as np

def cosine_similarity(v1, v2):
    """
    حساب التشابه بين vectorين
    النتيجة من -1 (عكس بعض) لـ 1 (متطابقين)
    """
    dot_product = np.dot(v1, v2)        # A · B
    norm_v1 = np.linalg.norm(v1)        # ||A||
    norm_v2 = np.linalg.norm(v2)        # ||B||
    return dot_product / (norm_v1 * norm_v2)

# ثلاث نصوص للمقارنة
text1 = "BIM is Building Information Modeling"
text2 = "Building Information Modeling is used in construction"
text3 = "I like pizza and pasta"

print("📝 النصوص:")
print(f"   1: {text1}")
print(f"   2: {text2}")
print(f"   3: {text3}")
print()
print("⏳ Converting to vectors...")

vec1 = embeddings.embed_query(text1)
vec2 = embeddings.embed_query(text2)
vec3 = embeddings.embed_query(text3)

sim_1_2 = cosine_similarity(vec1, vec2)
sim_1_3 = cosine_similarity(vec1, vec3)

print()
print("📊 النتائج:")
print(f"   التشابه بين 1 و 2: {sim_1_2:.2%} ✅ (متشابهين - كلهم عن BIM)")
print(f"   التشابه بين 1 و 3: {sim_1_3:.2%} ❌ (مختلفين - واحد BIM والتاني أكل)")

📝 النصوص:
   1: BIM is Building Information Modeling
   2: Building Information Modeling is used in construction
   3: I like pizza and pasta

⏳ Converting to vectors...

📊 النتائج:
   التشابه بين 1 و 2: 76.26% ✅ (متشابهين - كلهم عن BIM)
   التشابه بين 1 و 3: 49.90% ❌ (مختلفين - واحد BIM والتاني أكل)


### 📊 Similarity Thresholds

| النسبة | التفسير |
|--------|--------|
| **> 0.7** | متشابهين جداً ✅ |
| **0.5 - 0.7** | متعلقين ✅ |
| **< 0.5** | غالباً مش related ❌ |

---

# 📚 Section 3: Vector Database (ChromaDB)

## 💾 إيه هو الـ Vector Database?

قاعدة بيانات متخصصة في:
- **تخزين** الـ Vectors
- **البحث** عن الـ Vectors المتشابهة بسرعة

## 🗄️ ChromaDB

- مجاني ومفتوح المصدر
- سهل الاستخدام
- يشتغل locally (مش محتاج server)

In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  إنشاء Vector Database                                       ║
# ╚══════════════════════════════════════════════════════════════╝

from langchain_community.vectorstores import Chroma

# نصوص عن BIM (تخيل إنها أجزاء من PDF)
texts = [
    "BIM stands for Building Information Modeling. It is a digital representation of physical and functional characteristics of a facility.",
    "LOD means Level of Development. LOD 100 is conceptual, LOD 200 is approximate geometry, LOD 300 is precise geometry, LOD 400 is fabrication, LOD 500 is as-built.",
    "Revit is a BIM software developed by Autodesk. It is used for architectural design, structural engineering, and MEP coordination.",
    "IFC stands for Industry Foundation Classes. It is an open file format for BIM data exchange between different software applications.",
    "Clash detection is the process of identifying conflicts between different building systems like structural beams and MEP ducts before construction.",
]

print("⏳ Creating Vector Database...")
print("   (بيحول كل نص لـ vector ويخزنه)")

vectorstore = Chroma.from_texts(
    texts=texts,
    embedding=embeddings
)

print(f"\n✅ Vector Database ready!")
print(f"   Stored: {len(texts)} documents")

⏳ Creating Vector Database...
   (بيحول كل نص لـ vector ويخزنه)


NameError: name 'embeddings' is not defined

In [9]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  البحث في الـ Vector Database                                ║
# ║                                                              ║
# ║  similarity_search(query, k):                                ║
# ║    - query: السؤال                                          ║
# ║    - k: عدد النتائج المطلوبة                                 ║
# ╚══════════════════════════════════════════════════════════════╝

question = "What is LOD in BIM?"

print(f"❓ السؤال: {question}")
print("⏳ Searching...")

results = vectorstore.similarity_search(question, k=2)

print(f"\n📄 النتائج ({len(results)}):\n")
for i, doc in enumerate(results, 1):
    print(f"─── Result {i} ───")
    print(f"{doc.page_content}")
    print()

❓ السؤال: What is LOD in BIM?
⏳ Searching...

📄 النتائج (2):

─── Result 1 ───
LOD means Level of Development. LOD 100 is conceptual, LOD 200 is approximate geometry, LOD 300 is precise geometry, LOD 400 is fabrication, LOD 500 is as-built.

─── Result 2 ───
BIM stands for Building Information Modeling. It is a digital representation of physical and functional characteristics of a facility.



In [10]:
# تجربة أسئلة تانية

questions = [
    "What software is used for BIM?",
    "How to exchange BIM data between applications?",
    "What is clash detection?"
]

for q in questions:
    print(f"❓ {q}")
    result = vectorstore.similarity_search(q, k=1)[0]
    print(f"💬 {result.page_content[:100]}...")
    print()

❓ What software is used for BIM?
💬 Revit is a BIM software developed by Autodesk. It is used for architectural design, structural engin...

❓ How to exchange BIM data between applications?
💬 IFC stands for Industry Foundation Classes. It is an open file format for BIM data exchange between ...

❓ What is clash detection?
💬 Clash detection is the process of identifying conflicts between different building systems like stru...



---

# 📚 Section 4: Document Chunking

## ✂️ ليه بنقسم الـ Documents?

1. **LLM Token Limits**: مينفعش تبعت 100 صفحة مرة واحدة
2. **Better Search**: البحث في chunks صغيرة أدق

## 📏 إعدادات مهمة:

| Parameter | الوصف | قيمة مقترحة |
|-----------|-------|------------|
| `chunk_size` | حجم كل قطعة | 300-500 |
| `chunk_overlap` | التداخل بين القطع | 50-100 |

In [11]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Document Class                                              ║
# ║                                                              ║
# ║  Document له جزئين:                                          ║
# ║    - page_content: النص                                      ║
# ║    - metadata: معلومات إضافية (الصفحة، المصدر، إلخ)          ║
# ╚══════════════════════════════════════════════════════════════╝

from langchain_core.documents import Document

# إنشاء documents للتجربة
sample_documents = [
    Document(
        page_content="""Building Information Modeling (BIM) is a process involving the generation 
        and management of digital representations of physical and functional characteristics of places. 
        BIMs are files which can be extracted, exchanged or networked to support decision-making regarding 
        a built asset. BIM software is used by individuals, businesses and government agencies who plan, 
        design, construct, operate and maintain buildings and diverse physical infrastructures.""",
        metadata={"page": 1, "source": "BIM_Guide.pdf"}
    ),
    Document(
        page_content="""Level of Development (LOD) is a framework that enables AEC practitioners to specify 
        and articulate with a high level of clarity the content and reliability of Building Information Models. 
        LOD 100 represents a conceptual design. LOD 200 shows approximate geometry. LOD 300 provides precise 
        geometry suitable for construction documents. LOD 400 includes fabrication details. LOD 500 represents 
        the as-built condition of the facility.""",
        metadata={"page": 2, "source": "BIM_Guide.pdf"}
    ),
    Document(
        page_content="""Clash detection is a critical process in BIM coordination. It identifies conflicts 
        between different building systems before construction begins. For example, a structural beam might 
        conflict with an HVAC duct, or electrical conduits might pass through a structural column. Software 
        like Navisworks and Solibri can automatically detect these clashes, saving significant time and cost 
        during construction.""",
        metadata={"page": 3, "source": "BIM_Guide.pdf"}
    ),
]

print(f"✅ Created {len(sample_documents)} documents")
print(f"\n📄 Example document:")
print(f"   Content: {sample_documents[0].page_content[:80]}...")
print(f"   Metadata: {sample_documents[0].metadata}")

✅ Created 3 documents

📄 Example document:
   Content: Building Information Modeling (BIM) is a process involving the generation 
     ...
   Metadata: {'page': 1, 'source': 'BIM_Guide.pdf'}


In [12]:
# Cell 1: Install only what you need
!pip install -q langchain-text-splitters

In [13]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Text Splitter                                               ║
# ║                                                              ║
# ║  RecursiveCharacterTextSplitter:                             ║
# ║    - بيقسم بذكاء (بيحافظ على الجمل)                          ║
# ║    - بيستخدم separators: ["\n\n", "\n", " ", ""]             ║
# ╚══════════════════════════════════════════════════════════════╝

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,      # حجم كل chunk
    chunk_overlap=30,    # التداخل (عشان ما نفقدش السياق)
)

chunks = text_splitter.split_documents(sample_documents)

print(f"📄 Original documents: {len(sample_documents)}")
print(f"📝 After chunking: {len(chunks)} chunks")
print()
print("📝 First 3 chunks:")
for i, chunk in enumerate(chunks[:3], 1):
    print(f"\n─── Chunk {i} (Page {chunk.metadata['page']}) ───")
    print(f"{chunk.page_content[:150]}...")

📄 Original documents: 3
📝 After chunking: 10 chunks

📝 First 3 chunks:

─── Chunk 1 (Page 1) ───
Building Information Modeling (BIM) is a process involving the generation 
        and management of digital representations of physical and functiona...

─── Chunk 2 (Page 1) ───
BIMs are files which can be extracted, exchanged or networked to support decision-making regarding...

─── Chunk 3 (Page 1) ───
a built asset. BIM software is used by individuals, businesses and government agencies who plan,...


In [14]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  تخزين الـ Chunks في Vector Database                         ║
# ╚══════════════════════════════════════════════════════════════╝

print("⏳ Storing chunks in Vector Database...")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

print(f"✅ Stored {len(chunks)} chunks!")

⏳ Storing chunks in Vector Database...
✅ Stored 10 chunks!


---

# 📚 Section 5: بناء RAG كامل

## 🔗 ربط كل الأجزاء:

```
Question → Search → Get Chunks → Build Prompt → LLM → Answer
```

In [15]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  إعداد الـ LLM (Chat Model)                                  ║
# ╚══════════════════════════════════════════════════════════════╝

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.3,  # Lower = more focused, Higher = more creative
)

print("✅ LLM ready!")
print("   Model: gemini-2.0-flash")
print("   Temperature: 0.3")

✅ LLM ready!
   Model: gemini-2.0-flash
   Temperature: 0.3


In [16]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  RAG Function                                                ║
# ║                                                              ║
# ║  الخطوات:                                                    ║
# ║    1. ابحث عن الـ chunks المتعلقة                            ║
# ║    2. اجمعهم في context                                      ║
# ║    3. ابني الـ prompt                                        ║
# ║    4. ابعت للـ LLM                                           ║
# ║    5. رجّع الإجابة                                           ║
# ╚══════════════════════════════════════════════════════════════╝

def ask_documents(question: str, k: int = 3) -> dict:
    """
    اسأل سؤال واحصل على إجابة من الـ documents.
    
    Args:
        question: السؤال
        k: عدد الـ chunks المطلوبة
    
    Returns:
        dict with question, answer, and sources
    """
    # 1. البحث عن الـ chunks المتعلقة
    relevant_chunks = vectorstore.similarity_search(question, k=k)
    
    # 2. تجميع الـ context
    context = "\n\n".join([chunk.page_content for chunk in relevant_chunks])
    
    # 3. بناء الـ prompt
    prompt = f"""Use the following context to answer the question.
If you cannot find the answer in the context, say "I don't have enough information to answer this question."
Always cite which part of the context you used.

Context:
{context}

Question: {question}

Answer:"""
    
    # 4. إرسال للـ LLM
    response = llm.invoke(prompt)
    
    # 5. إرجاع النتيجة
    return {
        "question": question,
        "answer": response.content,
        "sources": relevant_chunks
    }

print("✅ RAG Function ready!")

✅ RAG Function ready!


In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Helper Function لعرض النتائج                                ║
# ╚══════════════════════════════════════════════════════════════╝

def display_result(result):
    print("=" * 60)
    print(f"❓ السؤال: {result['question']}")
    print("=" * 60)
    print(f"\n💬 الإجابة:\n")
    print(result['answer'])
    print("\n" + "-" * 60)
    print("📚 المصادر:")
    for i, doc in enumerate(result['sources'], 1):
        source = doc.metadata.get('source', 'Unknown')
        page = doc.metadata.get('page', '?')
        print(f"   {i}. [{source}, Page {page}]")
        print(f"      \"{doc.page_content[:80]}...\"")
    print("=" * 60)

In [5]:
# تجربة 1: سؤال عن BIM
question = "can i play pubg?"
result = ask_documents(question)
display_result(result)

NameError: name 'ask_documents' is not defined

In [ ]:
# تجربة 2: سؤال عن LOD

result = ask_documents("What are the different LOD levels and what do they represent?")
display_result(result)

In [ ]:
# تجربة 3: سؤال عن Clash Detection

result = ask_documents("What is clash detection and what software can I use for it?")
display_result(result)

In [ ]:
# تجربة 4: سؤال مش في الـ documents

result = ask_documents("What is the weather today?")
display_result(result)

---

# 📚 Section 6: Streamlit App

## 🌐 بناء واجهة مستخدم

هنعمل app بسيط للـ Chat with PDF

In [ ]:
# ==========================================
# BIM RAG App - Retrieval-Augmented Generation
# ==========================================
# This app allows users to upload BIM (Building Information Modeling) PDF documents,
# process them into searchable chunks, and ask questions about their content
# using Google's Gemini AI models.
#
# How RAG works:
# 1. User uploads a PDF document
# 2. The PDF is split into small text chunks
# 3. Each chunk is converted to a vector (embedding) using Gemini Embedding model
# 4. Vectors are stored in ChromaDB (vector database)
# 5. When user asks a question, the question is also converted to a vector
# 6. We find the most similar chunks to the question (similarity search)
# 7. Those chunks are sent as context to Gemini LLM to generate an answer

# ============ IMPORTS ============

import streamlit as st          # Web UI framework
import os                       # For file operations (deleting temp files)
import tempfile                 # For creating temporary files to save uploaded PDFs

# LangChain + Google Gemini integrations
from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,   # Converts text to vectors (embeddings)
    ChatGoogleGenerativeAI          # Chat model for generating answers
)

# LangChain document processing
from langchain_community.document_loaders import PyPDFLoader           # Reads PDF files and extracts text
from langchain_text_splitters import RecursiveCharacterTextSplitter    # Splits long text into smaller chunks
from langchain_community.vectorstores import Chroma                    # Vector database to store and search embeddings


# ============ PAGE CONFIGURATION ============
# Configure the Streamlit page layout and metadata

st.set_page_config(
    page_title="BIM RAG App",
    page_icon="📐",
    layout="wide",                      # Use full width of the browser
    initial_sidebar_state="expanded"    # Sidebar is open by default
)

st.title("📐 BIM RAG App")
st.caption("A simple Retrieval-Augmented Generation (RAG) app for Building Information Modeling (BIM) documents using Streamlit and LangChain.")


# ============ SIDEBAR - USER INPUTS ============
# The sidebar contains: API key input, file uploader, and action buttons

# Google API Key - needed to access Gemini models
api_key = st.sidebar.text_input("Enter your Google API Key", type="password")

# File uploader - accepts PDF files only, supports multiple files
st.sidebar.header("Upload BIM Documents")
uploaded_files = st.sidebar.file_uploader("Upload PDF files", type=["pdf"], accept_multiple_files=True)
if uploaded_files:
    st.sidebar.success(f"✅ {len(uploaded_files)} file(s) uploaded successfully!")

# Button to start processing the uploaded documents
process_btn = st.sidebar.button("Process Documents")

# Button to clear all documents and chat history from memory
if st.button("Clear Documents"):
    st.session_state.messages = []              # Clear chat history
    st.session_state.pop("vectorstore", None)   # Remove vector database
    st.session_state.pop("api_key", None)       # Remove saved API key
    st.success("✅ Documents cleared from session state!")
    st.rerun()  # Refresh the page


# ============ INITIALIZE SESSION STATE ============
# session_state persists data across Streamlit reruns (every interaction causes a rerun)
# We use it to store: chat messages, vector database, and API key

if "messages" not in st.session_state:
    st.session_state.messages = []


# ============ DOCUMENT PROCESSING ============
# This section runs when the user clicks "Process Documents"
# Steps: Save PDF to temp file -> Load PDF -> Split into chunks -> Create embeddings -> Store in ChromaDB

if process_btn and uploaded_files and api_key:
    with st.spinner("⏳ Loading and processing documents..."):

        # Step 1: Save the uploaded PDF to a temporary file
        # (PyPDFLoader needs a file path, not a file object)
        with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as temp_file:
            temp_file.write(uploaded_files[0].getvalue())  # Write the first uploaded file
            temp_file_path = temp_file.name

        # Step 2: Load the PDF and extract text from each page
        loader = PyPDFLoader(temp_file_path)
        pages = loader.load()  # Returns a list of Document objects, one per page

        # Step 3: Split pages into smaller chunks for better search accuracy
        # chunk_size=1000  -> each chunk is ~1000 characters
        # chunk_overlap=200 -> chunks overlap by 200 chars to avoid losing context at boundaries
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        chunks = splitter.split_documents(pages)
        st.info(f"📄 {len(pages)} pages → {len(chunks)} chunks")

        # Step 4: Create embeddings and store in ChromaDB
        # task_type="RETRIEVAL_DOCUMENT" -> optimized for storing documents (not queries)
        doc_embeddings = GoogleGenerativeAIEmbeddings(
            model="gemini-embedding-001",
            google_api_key=api_key,
            task_type="RETRIEVAL_DOCUMENT"
        )

        # Create the vector store from the document chunks
        vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=doc_embeddings,
            collection_name="bim_docs_7"
        )

        st.success("✅ Documents processed and vector store created successfully!")
        st.write(f"📚 Total documents in vector store: {vectorstore._collection.count()}")

        # Step 5: Save to session state so it persists across reruns
        st.session_state.vectorstore = vectorstore
        st.session_state.api_key = api_key

        # Step 6: Clean up the temporary file
        os.unlink(temp_file_path)

# Show warnings if user clicks Process without providing required inputs
elif process_btn:
    if not api_key:
        st.warning("⚠️ Please enter your Google API Key!")
    if not uploaded_files:
        st.warning("⚠️ Please upload at least one PDF file!")


# ============ DISPLAY CHAT HISTORY ============
# Show all previous messages (both user questions and assistant answers)
# This runs on every rerun to keep the chat visible

for msg in st.session_state.messages:
    st.chat_message(msg["role"]).write(msg["content"])


# ============ CHAT INPUT & RAG PIPELINE ============
# This section handles: receiving user questions, searching documents, and generating answers

# st.chat_input returns the user's message (or None if empty)
# The walrus operator (:=) assigns and checks in one line
if prompt := st.chat_input("Ask a question about your BIM documents..."):

    # Check if documents have been processed first
    if "vectorstore" not in st.session_state or "api_key" not in st.session_state:
        st.warning("⚠️ Please upload and process your BIM documents first!")
        st.stop()  # Stop execution here

    # Add user message to chat history and display it
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Generate the assistant's response
    with st.chat_message("assistant"):
        with st.spinner("⏳ Thinking..."):

            # --- RAG Step 1: Convert the question to a vector ---
            # task_type="RETRIEVAL_QUERY" -> optimized for search queries (not documents)
            # Note: we use embed_documents([prompt])[0] instead of embed_query(prompt)
            # because embed_query has a bug in langchain-google-genai 4.2.1
            query_embeddings = GoogleGenerativeAIEmbeddings(
                model="gemini-embedding-001",
                google_api_key=st.session_state.api_key,
                task_type="RETRIEVAL_QUERY"
            )
            query_vector = query_embeddings.embed_documents([prompt])[0]

            # --- RAG Step 2: Find the 3 most similar chunks to the question ---
            # similarity_search_by_vector compares the query vector against all stored vectors
            # k=3 means return the top 3 most relevant chunks
            docs = st.session_state.vectorstore.similarity_search_by_vector(query_vector, k=3)

            # --- RAG Step 3: Combine the relevant chunks into a single context string ---
            context = "\n\n".join([doc.page_content for doc in docs])

            # --- RAG Step 4: Send the question + context to Gemini LLM ---
            # temperature=0.3 -> lower = more focused/deterministic answers
            llm = ChatGoogleGenerativeAI(
                model="gemini-3-flash-preview",
                google_api_key=st.session_state.api_key,
                temperature=0.3
            )

            # Build the prompt: question + retrieved context
            full_prompt = f"""Use the following context to answer the question. If you can not find the answer, say so.

Question: {prompt}

Context:
{context}

Answer:"""

            # --- RAG Step 5: Get and display the answer ---
            response = llm.invoke(full_prompt)

            # Handle response - Gemini 3 returns content as a list, older models return a string
            if isinstance(response.content, list):
                answer = "".join(
                    [c.get("text", "") if isinstance(c, dict) else str(c) for c in response.content]
                ).strip()
            else:
                answer = response.content.strip()

            # Display the answer
            st.markdown(answer)

            # --- Show source documents used to generate the answer ---
            with st.expander("📚 Source Documents"):
                for i, doc in enumerate(docs, 1):
                    source = doc.metadata.get('source', 'Unknown Source')
                    page = doc.metadata.get('page', 'N/A')
                    st.markdown(f"**Source {i}:** {source} - Page {page}")
                    st.markdown(f"```{doc.page_content[:250]}...```")

            # Save the assistant's answer to chat history
            st.session_state.messages.append({"role": "assistant", "content": answer})


---

# 🔬 Section 7: Lab Project

## 🎯 المشروع: BIM Document Assistant

### 📋 المتطلبات:

1. **Upload PDF** - ارفع PDF عن BIM (Standards, Specifications, etc.)
2. **Process** - قسّمه لـ chunks وخزّنه
3. **Chat** - اسأل أسئلة واحصل على إجابات
4. **Sources** - اعرض المصادر

### 🏆 Bonus Features:

- [ ] دعم أكثر من PDF
- [ ] تصدير المحادثة
- [ ] دعم اللغة العربية

---

## 📝 خطوات التنفيذ:

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Lab Step 1: Load PDF                                        ║
# ║                                                              ║
# ║  غيّر المسار لـ PDF عندك                                     ║
# ╚══════════════════════════════════════════════════════════════╝

from langchain_community.document_loaders import PyPDFLoader

# ⚠️ غيّر المسار ده لـ PDF عندك
PDF_PATH = "your_document.pdf"  # مثال: "BIM_Standards.pdf"

# لو مش عندك PDF، هنستخدم Sample Text
USE_SAMPLE = True  # غيّرها لـ False لو عندك PDF

if USE_SAMPLE:
    print("📝 Using sample documents (no PDF uploaded)")
    
    from langchain_core.documents import Document
    
    documents = [
        Document(
            page_content="""BIM Execution Plan (BEP) is a document that outlines how the project team 
            will implement BIM on a project. It includes BIM goals, uses, process maps, and information 
            exchange requirements. The BEP should be developed at the start of the project and updated 
            throughout the project lifecycle.""",
            metadata={"page": 1, "source": "BIM_Guide.pdf"}
        ),
        Document(
            page_content="""Model Element Author (MEA) is the party responsible for developing the content 
            of a specific model element to the LOD required. The MEA is typically identified in the BIM 
            Execution Plan and may change as the project progresses through different phases.""",
            metadata={"page": 2, "source": "BIM_Guide.pdf"}
        ),
        Document(
            page_content="""Common Data Environment (CDE) is a single source of information for the project, 
            used to collect, manage and disseminate documentation, the graphical model and non-graphical data. 
            All project participants can access the CDE to ensure everyone is working with the latest information.""",
            metadata={"page": 3, "source": "BIM_Guide.pdf"}
        ),
        Document(
            page_content="""Federated Model is a combined model created by linking separate discipline models 
            together. Unlike a merged model, the individual models remain separate files. This approach allows 
            for better collaboration as each discipline can work on their model independently.""",
            metadata={"page": 4, "source": "BIM_Guide.pdf"}
        ),
        Document(
            page_content="""4D BIM adds the dimension of time to the 3D model, enabling project scheduling 
            and construction sequencing visualization. 5D BIM adds cost information, allowing for real-time 
            cost estimation and budget tracking. 6D BIM includes facility management information for the 
            operational phase of the building.""",
            metadata={"page": 5, "source": "BIM_Guide.pdf"}
        ),
    ]
    print(f"✅ Loaded {len(documents)} sample documents")
    
else:
    # Load actual PDF
    print(f"📄 Loading PDF: {PDF_PATH}")
    loader = PyPDFLoader(PDF_PATH)
    documents = loader.load()
    print(f"✅ Loaded {len(documents)} pages")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Lab Step 2: Split Documents                                 ║
# ╚══════════════════════════════════════════════════════════════╝

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)

print(f"📄 Original: {len(documents)} documents")
print(f"📝 Chunks: {len(chunks)}")
print(f"\n📊 Stats:")
lengths = [len(c.page_content) for c in chunks]
print(f"   Min chunk: {min(lengths)} chars")
print(f"   Max chunk: {max(lengths)} chars")
print(f"   Avg chunk: {sum(lengths)//len(lengths)} chars")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Lab Step 3: Create Vector Database                          ║
# ╚══════════════════════════════════════════════════════════════╝

from langchain_community.vectorstores import Chroma

print("⏳ Creating Vector Database...")

lab_vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="bim_assistant"
)

print(f"✅ Vector Database ready with {len(chunks)} chunks!")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Lab Step 4: Create the Assistant                            ║
# ╚══════════════════════════════════════════════════════════════╝

class BIMAssistant:
    """
    BIM Document Assistant
    يقدر يجاوب على أسئلة من الـ documents
    """
    
    def __init__(self, vectorstore, llm):
        self.vectorstore = vectorstore
        self.llm = llm
        self.chat_history = []
    
    def ask(self, question: str, k: int = 3) -> str:
        """
        اسأل سؤال واحصل على إجابة
        """
        # Search
        docs = self.vectorstore.similarity_search(question, k=k)
        
        # Build context
        context = "\n\n".join([d.page_content for d in docs])
        
        # Build prompt
        prompt = f"""You are a helpful BIM assistant. Use the following context to answer the question.
Be concise but comprehensive. If you cannot find the answer, say so.

Context:
{context}

Question: {question}

Answer:"""
        
        # Get response
        response = self.llm.invoke(prompt)
        answer = response.content
        
        # Save to history
        self.chat_history.append({
            "question": question,
            "answer": answer,
            "sources": [d.metadata for d in docs]
        })
        
        return answer
    
    def get_history(self):
        """احصل على تاريخ المحادثة"""
        return self.chat_history
    
    def clear_history(self):
        """امسح تاريخ المحادثة"""
        self.chat_history = []

# Create the assistant
assistant = BIMAssistant(lab_vectorstore, llm)
print("✅ BIM Assistant ready!")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Lab Step 5: Test the Assistant                              ║
# ╚══════════════════════════════════════════════════════════════╝

# اختبر الـ Assistant

print("🤖 BIM Assistant")
print("=" * 50)

questions = [
    "What is a BIM Execution Plan?",
    "What is a Common Data Environment?",
    "What are the different dimensions of BIM (4D, 5D, 6D)?",
]

for q in questions:
    print(f"\n❓ {q}")
    print("-" * 50)
    answer = assistant.ask(q)
    print(f"💬 {answer}")
    print()

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Lab Step 6: Interactive Chat                                ║
# ╚══════════════════════════════════════════════════════════════╝

# Interactive chat loop
print("🤖 BIM Assistant - Interactive Mode")
print("=" * 50)
print("اكتب سؤالك واضغط Enter")
print("اكتب 'quit' للخروج")
print("=" * 50)

while True:
    question = input("\n❓ You: ").strip()
    
    if question.lower() in ['quit', 'exit', 'q']:
        print("\n👋 Goodbye!")
        break
    
    if not question:
        continue
    
    answer = assistant.ask(question)
    print(f"\n💬 Assistant: {answer}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Lab Step 7: Export Chat History                             ║
# ╚══════════════════════════════════════════════════════════════╝

import json
from datetime import datetime

def export_history(assistant, filename=None):
    """تصدير تاريخ المحادثة لملف JSON"""
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"chat_history_{timestamp}.json"
    
    history = assistant.get_history()
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    
    print(f"✅ Exported {len(history)} conversations to {filename}")
    return filename

# Export
if assistant.get_history():
    export_history(assistant)
else:
    print("📝 No chat history to export yet.")

Section 8: Lab Exercise

### المطلوب: بناء RAG app متخصص

**اختار واحد:**
- 📐 BIM Standards Assistant
- 🏗️ Building Code Helper
- 🦺 Safety Regulations Bot

---

## 📚 Resources

| Resource | Link |
|----------|------|
| LangChain Docs | https://python.langchain.com |
| ChromaDB | https://docs.trychroma.com |
| Google AI | https://ai.google.dev |
| Course | https://www.deeplearning.ai/short-courses/langchain-chat-with-your-data/ |

---

## ✅ Day 4 Complete!

النهارده اتعلمنا:
- [x] إيه هو RAG ومحتاجينه ليه
- [x] Embeddings وإزاي بتشتغل
- [x] ChromaDB للتخزين والبحث
- [x] تقسيم النصوص لـ chunks
- [x] بناء RAG Function كامل
- [x] Streamlit App

---

🎯 **Next: Day 5 - AI Agents**